## AR / MA / ARMA / ARIMA / SARIMA 

This notebook runs AR($p$) (linear regression on lags) and statsmodels MA/ARMA/ARIMA/SARIMA models using walk-forward CV.

## MA($q$)

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.optimize import minimize
from scipy.signal import lfilter
from sklearn.metrics import mean_squared_error
from tqdm import tqdm
import warnings

# --- 1. CLASSE MA(q) FROM SCRATCH ---

class SimpleMA:
    def __init__(self, q=1):
        self.q = int(q)
        self.mu = 0.0
        self.thetas = np.zeros(self.q)
        self.last_resids = np.zeros(self.q)

    def _objective(self, params, y):
        mu = params[0]
        # On construit le filtre : [1, theta_1, theta_2, ..., theta_q]
        a = np.insert(params[1:], 0, 1.0)
        resid = lfilter([1.0], a, y - mu)
        
        sse = np.sum(resid**2)
        
        # GARDE-FOU 1 : Si les erreurs explosent ou donnent NaN, on renvoie 
        # une pénalité gigantesque pour forcer l'optimiseur à faire demi-tour.
        if not np.isfinite(sse) or sse > 1e10:
            return 1e10
            
        return sse

    def fit(self, y):
        initial_guess = np.zeros(self.q + 1)
        initial_guess[0] = np.mean(y)
        
        # GARDE-FOU 2 : On limite chaque theta entre -0.99 et 0.99 
        # (None, None) est pour la moyenne 'mu' qui n'a pas de limite
        bounds = [(None, None)] + [(-0.99, 0.99)] * self.q
        
        # GARDE-FOU 3 : options={'maxiter': 50} empêche les boucles infinies
        res = minimize(
            self._objective, 
            initial_guess, 
            args=(y,), 
            method='L-BFGS-B',
            bounds=bounds,
            options={'maxiter': 50, 'ftol': 1e-4} # ftol permet d'être moins exigeant sur la précision
        )
        
        self.mu = res.x[0]
        self.thetas = res.x[1:]
        
        # Recalcul final propre
        a = np.insert(self.thetas, 0, 1.0)
        final_resids = lfilter([1.0], a, y - self.mu)
        
        # On stocke les q dernières erreurs pour la prédiction
        if len(final_resids) >= self.q:
            self.last_resids = final_resids[-self.q:]
        else:
            pad = np.zeros(self.q - len(final_resids))
            self.last_resids = np.concatenate([pad, final_resids])
            
        return self

    def predict(self, steps=1):
        preds = np.zeros(steps)
        e_array = np.concatenate([self.last_resids, np.zeros(steps)])
        
        for h in range(steps):
            sum_ma = 0
            for i in range(1, self.q + 1):
                sum_ma += self.thetas[i-1] * e_array[self.q + h - i]
            
            preds[h] = self.mu + sum_ma
            
        return preds

# --- 2. WALK-FORWARD CROSS VALIDATION ---

def wfcv_ma(y, q=1, step_size=50, fold_size=200):
    y_vals = y.values
    n = len(y_vals)
    
    preds = []
    truths = []
    mse_tab = []
    
    model = SimpleMA(q=q)
    
    for start in range(0, n - fold_size - step_size + 1, step_size):
        y_train = y_vals[start : start + fold_size]
        y_test = y_vals[start + fold_size : start + fold_size + step_size]
        
        try:
            model.fit(y_train)
            fcst = model.predict(steps=len(y_test))
            mse_tab.append(mean_squared_error(y_test, fcst))
        except Exception:
            fcst = [np.nan] * len(y_test)
            mse_tab.append(np.nan)
            
        preds.extend(fcst)
        truths.extend(y_test)
        
    return np.array(preds), np.array(truths), np.array(mse_tab)


# --- 3. FONCTION D'ÉVALUATION PAR TICKER ---

def extract(df_all, ticker):
    df_t = df_all[['Date', ticker]].copy()
    df_t = df_t.rename(columns={ticker: 'return'})
    return df_t

def stats_forecasting_ma_custom(name_ticker, df_all, q=1, step_size=50, fold_size=200):
    df_t = extract(df_all, name_ticker)
    
    df_t = df_t[df_t['return'] > -1].copy()
    df_t['log_return'] = np.log1p(df_t['return'])
    df_t['log_return'] = df_t['log_return'].replace([np.inf, -np.inf], np.nan)
    df_t = df_t.dropna(subset=['log_return'])
    
    y = df_t['log_return']
    
    y_pred, y_true, mse_tab = wfcv_ma(y=y, q=q, step_size=step_size, fold_size=fold_size)

    mask = np.isfinite(y_pred) & np.isfinite(y_true)
    if mask.sum() >= 3:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            reg = sm.OLS(y_true[mask], sm.add_constant(y_pred[mask])).fit()
            ols_r2 = float(reg.rsquared)
            ols_intercept = float(reg.params[0])
            ols_slope = float(reg.params[1])
            p_int = float(reg.pvalues[0])
            p_slope = float(reg.pvalues[1])
    else:
        ols_r2 = ols_intercept = ols_slope = p_int = p_slope = float('nan')

    return {
        'SYMBOL': name_ticker,
        'Model': f'MA({q}) Custom',
        'q': q,
        'MSE': float(np.nanmean(mse_tab)) if len(mse_tab) > 0 else np.nan,
        'OLS_R2': ols_r2,
        'OLS_Intercept': ols_intercept,
        'OLS_Slope': ols_slope,
        'OLS_P_Value_Intercept': p_int,
        'P_Value_Slope': p_slope,
        'n_obs': int(len(y)),
    }


# --- 4. EXÉCUTION PRINCIPALE ---

if __name__ == "__main__":
    STEP_SIZE = 50
    FOLD_SIZE = 200

    # Charge le DataFrame UNE SEULE FOIS
    df_all = pd.read_csv('Datasets/returns_all.csv')
    df_all['Date'] = pd.to_datetime(df_all['Date'], errors='coerce')
    
    all_tickers = df_all.columns[1:].tolist()

    ma_qs = [2, 5, 10]
    results_ma = {}

    # Boucle sur les différents MA(q)
    for q in tqdm(ma_qs, desc='Boucle MA(q) — différents q'):
        rows = []
        
        # Boucle sur tous les tickers pour un q donné
        for name in tqdm(all_tickers, desc=f'MA({q}) — all tickers', leave=False):
            res = stats_forecasting_ma_custom(
                name_ticker=name, 
                df_all=df_all, 
                q=q,               # <-- On passe le paramètre q ici
                step_size=STEP_SIZE, 
                fold_size=FOLD_SIZE
            )
            rows.append(res)
            
        # Sauvegarde des résultats pour ce q
        df_q = pd.DataFrame(rows)
        results_ma[q] = df_q
        df_q.to_csv(f"Resultats/resultats_all_tickers_MA({q})_custom.csv", index=False)
        
    print("\nTous les calculs MA(q) sont terminés avec succès !")

Boucle MA(q) — différents q: 100%|██████████| 3/3 [16:59<00:00, 339.89s/it]


Tous les calculs MA(q) sont terminés avec succès !


## ARMA($p, q$)

In [2]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.optimize import minimize
from scipy.signal import lfilter
from sklearn.metrics import mean_squared_error
from tqdm import tqdm
import warnings

# --- 1. CLASSE ARMA(p, q) FROM SCRATCH ---

class SimpleARMA:
    def __init__(self, p=1, q=1):
        self.p = int(p)
        self.q = int(q)
        self.mu = 0.0
        self.phis = np.zeros(self.p)    # Coefficients AR
        self.thetas = np.zeros(self.q)  # Coefficients MA
        self.last_y_centered = np.zeros(self.p) # Historique des Y (centrés)
        self.last_resids = np.zeros(self.q)     # Historique des erreurs

    def _objective(self, params, y):
        mu = params[0]
        phis = params[1 : 1 + self.p]
        thetas = params[1 + self.p :]
        
        # Filtre lfilter(b, a, x) :
        # b (numérateur)   = [1, -phi_1, -phi_2, ...]   => Partie AR
        # a (dénominateur) = [1, theta_1, theta_2, ...] => Partie MA
        b = np.insert(-phis, 0, 1.0)
        a = np.insert(thetas, 0, 1.0)
        
        # Le filtre calcule directement les résidus
        resid = lfilter(b, a, y - mu)
        
        sse = np.sum(resid**2)
        
        # GARDE-FOU : On bloque les explosions exponentielles
        if not np.isfinite(sse) or sse > 1e10:
            return 1e10
            
        return sse

    def fit(self, y):
        # Initialisation : mu = moyenne, phis = 0, thetas = 0
        initial_guess = np.zeros(1 + self.p + self.q)
        initial_guess[0] = np.mean(y)
        
        # GARDE-FOU : On garde les phi et theta entre -0.99 et +0.99 pour la stabilité
        bounds = [(None, None)] + [(-0.99, 0.99)] * (self.p + self.q)
        
        res = minimize(
            self._objective, 
            initial_guess, 
            args=(y,), 
            method='L-BFGS-B',
            bounds=bounds,
            options={'maxiter': 50, 'ftol': 1e-4}
        )
        
        self.mu = res.x[0]
        self.phis = res.x[1 : 1 + self.p]
        self.thetas = res.x[1 + self.p :]
        
        # --- Recalcul et stockage pour la prédiction ---
        y_centered = y - self.mu
        b = np.insert(-self.phis, 0, 1.0)
        a = np.insert(self.thetas, 0, 1.0)
        final_resids = lfilter(b, a, y_centered)
        
        # On sauvegarde les p derniers (Y - mu)
        if self.p > 0:
            if len(y_centered) >= self.p:
                self.last_y_centered = y_centered[-self.p:]
            else:
                self.last_y_centered = np.concatenate([np.zeros(self.p - len(y_centered)), y_centered])
                
        # On sauvegarde les q dernières erreurs
        if self.q > 0:
            if len(final_resids) >= self.q:
                self.last_resids = final_resids[-self.q:]
            else:
                self.last_resids = np.concatenate([np.zeros(self.q - len(final_resids)), final_resids])
            
        return self

    def predict(self, steps=1):
        preds = np.zeros(steps)
        
        # On crée des tableaux glissants pour piocher le passé et écrire le futur
        y_hist = np.concatenate([self.last_y_centered, np.zeros(steps)])
        e_hist = np.concatenate([self.last_resids, np.zeros(steps)])
        
        for h in range(steps):
            # Calcul de la partie Autoregressive (AR)
            ar_sum = 0
            for i in range(1, self.p + 1):
                ar_sum += self.phis[i-1] * y_hist[self.p + h - i]
                
            # Calcul de la partie Moyenne Mobile (MA)
            ma_sum = 0
            for j in range(1, self.q + 1):
                ma_sum += self.thetas[j-1] * e_hist[self.q + h - j]
            
            # Nouvelle valeur centrée prédite
            pred_centered = ar_sum + ma_sum
            
            # On stocke pour l'itération suivante (l'erreur future reste 0)
            y_hist[self.p + h] = pred_centered
            
            # La prédiction finale est la valeur centrée + la moyenne
            preds[h] = self.mu + pred_centered
            
        return preds


# --- 2. WALK-FORWARD CROSS VALIDATION ---

def wfcv_arma(y, p=1, q=1, step_size=50, fold_size=200):
    y_vals = y.values
    n = len(y_vals)
    
    preds = []
    truths = []
    mse_tab = []
    
    model = SimpleARMA(p=p, q=q)
    
    for start in range(0, n - fold_size - step_size + 1, step_size):
        y_train = y_vals[start : start + fold_size]
        y_test = y_vals[start + fold_size : start + fold_size + step_size]
        
        try:
            model.fit(y_train)
            fcst = model.predict(steps=len(y_test))
            mse_tab.append(mean_squared_error(y_test, fcst))
        except Exception:
            fcst = [np.nan] * len(y_test)
            mse_tab.append(np.nan)
            
        preds.extend(fcst)
        truths.extend(y_test)
        
    return np.array(preds), np.array(truths), np.array(mse_tab)


# --- 3. FONCTION D'ÉVALUATION PAR TICKER ---

def extract(df_all, ticker):
    df_t = df_all[['Date', ticker]].copy()
    df_t = df_t.rename(columns={ticker: 'return'})
    return df_t

def stats_forecasting_arma_custom(name_ticker, df_all, p=1, q=1, step_size=50, fold_size=200):
    df_t = extract(df_all, name_ticker)
    
    df_t = df_t[df_t['return'] > -1].copy()
    df_t['log_return'] = np.log1p(df_t['return'])
    df_t['log_return'] = df_t['log_return'].replace([np.inf, -np.inf], np.nan)
    df_t = df_t.dropna(subset=['log_return'])
    
    y = df_t['log_return']
    
    y_pred, y_true, mse_tab = wfcv_arma(y=y, p=p, q=q, step_size=step_size, fold_size=fold_size)

    mask = np.isfinite(y_pred) & np.isfinite(y_true)
    if mask.sum() >= 3:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            reg = sm.OLS(y_true[mask], sm.add_constant(y_pred[mask])).fit()
            ols_r2 = float(reg.rsquared)
            ols_intercept = float(reg.params[0])
            ols_slope = float(reg.params[1])
            p_int = float(reg.pvalues[0])
            p_slope = float(reg.pvalues[1])
    else:
        ols_r2 = ols_intercept = ols_slope = p_int = p_slope = float('nan')

    return {
        'SYMBOL': name_ticker,
        'Model': f'ARMA({p},{q}) Custom',
        'p': p,
        'q': q,
        'MSE': float(np.nanmean(mse_tab)) if len(mse_tab) > 0 else np.nan,
        'OLS_R2': ols_r2,
        'OLS_Intercept': ols_intercept,
        'OLS_Slope': ols_slope,
        'OLS_P_Value_Intercept': p_int,
        'P_Value_Slope': p_slope,
        'n_obs': int(len(y)),
    }


# --- 4. EXÉCUTION PRINCIPALE ---

if __name__ == "__main__":
    STEP_SIZE = 50
    FOLD_SIZE = 200

    # Charge le DataFrame UNE SEULE FOIS
    df_all = pd.read_csv('Datasets/returns_all.csv')
    df_all['Date'] = pd.to_datetime(df_all['Date'], errors='coerce')
    
    all_tickers = df_all.columns[1:].tolist()

    # Liste des combinaisons (p, q) que tu souhaites tester
    # Par exemple: ARMA(1,1), ARMA(2,2), ARMA(5,5)
    arma_orders = [(1, 1), (2, 2), (5, 5)]
    results_arma = {}

    for p, q in tqdm(arma_orders, desc='Boucle ARMA(p,q)'):
        rows = []
        
        for name in tqdm(all_tickers, desc=f'ARMA({p},{q}) — all tickers', leave=False):
            res = stats_forecasting_arma_custom(
                name_ticker=name, 
                df_all=df_all, 
                p=p, 
                q=q,
                step_size=STEP_SIZE, 
                fold_size=FOLD_SIZE
            )
            rows.append(res)
            
        df_pq = pd.DataFrame(rows)
        results_arma[(p, q)] = df_pq
        df_pq.to_csv(f"Resultats/resultats_all_tickers_ARMA({p},{q})_custom.csv", index=False)
        
    print("\nTous les calculs ARMA(p,q) sont terminés avec succès !")

Boucle ARMA(p,q): 100%|██████████| 3/3 [15:01<00:00, 300.58s/it]


Tous les calculs ARMA(p,q) sont terminés avec succès !


## ARIMA($p, d, q$)

In [3]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.optimize import minimize
from scipy.signal import lfilter
from sklearn.metrics import mean_squared_error
from tqdm import tqdm
import warnings
import os

# --- 1. CLASSE ARMA (Le Moteur Mathématique) 

class SimpleARMA:
    def __init__(self, p=1, q=1):
        self.p = int(p)
        self.q = int(q)
        self.mu = 0.0
        self.phis = np.zeros(self.p)
        self.thetas = np.zeros(self.q)
        self.last_y_centered = np.zeros(self.p)
        self.last_resids = np.zeros(self.q)

    def _objective(self, params, y):
        mu = params[0]
        phis = params[1 : 1 + self.p]
        thetas = params[1 + self.p :]
        
        b = np.insert(-phis, 0, 1.0)
        a = np.insert(thetas, 0, 1.0)
        
        resid = lfilter(b, a, y - mu)
        sse = np.sum(resid**2)
        
        if not np.isfinite(sse) or sse > 1e10:
            return 1e10
            
        return sse

    def fit(self, y):
        initial_guess = np.zeros(1 + self.p + self.q)
        initial_guess[0] = np.mean(y)
        bounds = [(None, None)] + [(-0.99, 0.99)] * (self.p + self.q)
        
        res = minimize(
            self._objective, 
            initial_guess, 
            args=(y,), 
            method='L-BFGS-B',
            bounds=bounds,
            options={'maxiter': 50, 'ftol': 1e-4}
        )
        
        self.mu = res.x[0]
        self.phis = res.x[1 : 1 + self.p]
        self.thetas = res.x[1 + self.p :]
        
        y_centered = y - self.mu
        b = np.insert(-self.phis, 0, 1.0)
        a = np.insert(self.thetas, 0, 1.0)
        final_resids = lfilter(b, a, y_centered)
        
        if self.p > 0:
            if len(y_centered) >= self.p:
                self.last_y_centered = y_centered[-self.p:]
            else:
                self.last_y_centered = np.concatenate([np.zeros(self.p - len(y_centered)), y_centered])
                
        if self.q > 0:
            if len(final_resids) >= self.q:
                self.last_resids = final_resids[-self.q:]
            else:
                self.last_resids = np.concatenate([np.zeros(self.q - len(final_resids)), final_resids])
            
        return self

    def predict(self, steps=1):
        preds = np.zeros(steps)
        y_hist = np.concatenate([self.last_y_centered, np.zeros(steps)])
        e_hist = np.concatenate([self.last_resids, np.zeros(steps)])
        
        for h in range(steps):
            ar_sum = 0
            for i in range(1, self.p + 1):
                ar_sum += self.phis[i-1] * y_hist[self.p + h - i]
                
            ma_sum = 0
            for j in range(1, self.q + 1):
                ma_sum += self.thetas[j-1] * e_hist[self.q + h - j]
            
            pred_centered = ar_sum + ma_sum
            y_hist[self.p + h] = pred_centered
            preds[h] = self.mu + pred_centered
            
        return preds


# --- 2. CLASSE ARIMA (Le Wrapper de Différenciation) ---

class SimpleARIMA:
    def __init__(self, p=1, d=0, q=1):
        self.p = int(p)
        self.d = int(d)
        self.q = int(q)
        self.arma_model = SimpleARMA(p=self.p, q=self.q)
        self.last_y_history = np.array([]) 

    def fit(self, y):
        if len(y) <= self.d:
            raise ValueError(f"Pas assez de données pour différencier {self.d} fois.")
            
        if self.d > 0:
            self.last_y_history = y[-self.d:].copy()
            
        y_diff = y.copy()
        for _ in range(self.d):
            y_diff = np.diff(y_diff)
            
        self.arma_model.fit(y_diff)
        return self

    def predict(self, steps=1):
        diff_preds = self.arma_model.predict(steps=steps)
        
        if self.d == 0:
            return diff_preds
            
        last_vals = []
        current_series = self.last_y_history.copy()
        for _ in range(self.d):
            last_vals.append(current_series[-1])
            if len(current_series) > 1:
                current_series = np.diff(current_series)
                
        preds = np.array(diff_preds)
        for i in range(self.d - 1, -1, -1):
            preds = last_vals[i] + np.cumsum(preds)
            
        return preds


# --- 3. WALK-FORWARD CROSS VALIDATION ---

def wfcv_arima(y, p=1, d=0, q=1, step_size=50, fold_size=200):
    y_vals = y.values
    n = len(y_vals)
    
    preds = []
    truths = []
    mse_tab = []
    
    model = SimpleARIMA(p=p, d=d, q=q)
    
    for start in range(0, n - fold_size - step_size + 1, step_size):
        y_train = y_vals[start : start + fold_size]
        y_test = y_vals[start + fold_size : start + fold_size + step_size]
        
        try:
            model.fit(y_train)
            fcst = model.predict(steps=len(y_test))
            mse_tab.append(mean_squared_error(y_test, fcst))
        except Exception:
            fcst = [np.nan] * len(y_test)
            mse_tab.append(np.nan)
            
        preds.extend(fcst)
        truths.extend(y_test)
        
    return np.array(preds), np.array(truths), np.array(mse_tab)


# --- 4. FONCTION D'ÉVALUATION PAR TICKER ---

def extract(df_all, ticker):
    df_t = df_all[['Date', ticker]].copy()
    df_t = df_t.rename(columns={ticker: 'return'})
    return df_t

def stats_forecasting_arima_custom(name_ticker, df_all, p=1, d=0, q=1, step_size=50, fold_size=200):
    df_t = extract(df_all, name_ticker)
    
    df_t = df_t[df_t['return'] > -1].copy()
    df_t['log_return'] = np.log1p(df_t['return'])
    df_t['log_return'] = df_t['log_return'].replace([np.inf, -np.inf], np.nan)
    df_t = df_t.dropna(subset=['log_return'])
    
    y = df_t['log_return']
    
    y_pred, y_true, mse_tab = wfcv_arima(y=y, p=p, d=d, q=q, step_size=step_size, fold_size=fold_size)

    mask = np.isfinite(y_pred) & np.isfinite(y_true)
    if mask.sum() >= 3:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            reg = sm.OLS(y_true[mask], sm.add_constant(y_pred[mask])).fit()
            ols_r2 = float(reg.rsquared)
            ols_intercept = float(reg.params[0])
            ols_slope = float(reg.params[1])
            p_int = float(reg.pvalues[0])
            p_slope = float(reg.pvalues[1])
    else:
        ols_r2 = ols_intercept = ols_slope = p_int = p_slope = float('nan')

    return {
        'SYMBOL': name_ticker,
        'Model': f'ARIMA({p},{d},{q}) Custom',
        'p': p,
        'd': d,
        'q': q,
        'MSE': float(np.nanmean(mse_tab)) if len(mse_tab) > 0 else np.nan,
        'OLS_R2': ols_r2,
        'OLS_Intercept': ols_intercept,
        'OLS_Slope': ols_slope,
        'OLS_P_Value_Intercept': p_int,
        'P_Value_Slope': p_slope,
        'n_obs': int(len(y)),
    }


# --- 5. EXÉCUTION PRINCIPALE ---

if __name__ == "__main__":
    STEP_SIZE = 50
    FOLD_SIZE = 200

    # 1. Chargement unique des données
    df_all = pd.read_csv('Datasets/returns_all.csv')
    df_all['Date'] = pd.to_datetime(df_all['Date'], errors='coerce')
    all_tickers = df_all.columns[1:].tolist()

    # 2. Définition des modèles à tester
    arima_orders = [(1, 1, 1), (2, 1, 2)]
    
    # 3. Préparation du fichier de sauvegarde
    os.makedirs('Resultats', exist_ok=True)
    output_file = "Resultats/resultats_arima_all_tickers.csv"
    first_write = True

    # 4. Boucle PRINCIPALE sur les Tickers (La barre tqdm te montrera l'avancée globale)
    for name in tqdm(all_tickers, desc='Progression globale (Tickers)'):
        rows_for_ticker = []
        
        # Pour chaque ticker, on teste tous les paramètres ARIMA demandés
        for p, d, q in arima_orders:
            res = stats_forecasting_arima_custom(
                name_ticker=name, 
                df_all=df_all, 
                p=p, 
                d=d,
                q=q,
                step_size=STEP_SIZE, 
                fold_size=FOLD_SIZE
            )
            rows_for_ticker.append(res)
            
        # 5. Sauvegarde INCRÉMENTALE
        df_ticker = pd.DataFrame(rows_for_ticker)
        
        if first_write:
            # On crée le fichier avec les en-têtes
            df_ticker.to_csv(output_file, index=False, mode='w')
            first_write = False
        else:
            # On ajoute les lignes sans réécrire les en-têtes
            df_ticker.to_csv(output_file, index=False, mode='a', header=False)
            
    print(f"\nTous les calculs sont terminés ! Résultats consolidés dans : {output_file}")

Progression globale (Tickers): 100%|██████████| 1066/1066 [09:24<00:00,  1.89it/s]


Tous les calculs sont terminés ! Résultats consolidés dans : Resultats/resultats_arima_all_tickers.csv


## SARIMA($p$, $d$, $q$)($P$, $D$, $Q$, $S$)

In [5]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.optimize import minimize
from scipy.signal import lfilter
from sklearn.metrics import mean_squared_error
from tqdm import tqdm
import warnings
import os

# --- 1. CLASSE SARMA (Le Moteur Mathématique Saisonnier) ---

class SimpleSARMA:
    def __init__(self, p=1, q=1, P=0, Q=0, s=1):
        self.p = int(p)
        self.q = int(q)
        self.P = int(P)
        self.Q = int(Q)
        self.s = int(s)
        
        self.mu = 0.0
        self.phis = np.zeros(self.p)
        self.thetas = np.zeros(self.q)
        self.Phis = np.zeros(self.P)
        self.Thetas = np.zeros(self.Q)
        
        self.max_ar_lag = self.p + (self.P * self.s)
        self.max_ma_lag = self.q + (self.Q * self.s)
        
        self.last_y_centered = np.zeros(max(1, self.max_ar_lag))
        self.last_resids = np.zeros(max(1, self.max_ma_lag))

    def _build_polynomials(self, phis, thetas, Phis, Thetas):
        ar_poly = np.insert(-phis, 0, 1.0) if self.p > 0 else np.array([1.0])
        
        sar_poly = np.zeros(self.P * self.s + 1)
        sar_poly[0] = 1.0
        for i in range(1, self.P + 1):
            sar_poly[i * self.s] = -Phis[i-1]
            
        full_ar = np.polymul(ar_poly, sar_poly)
        
        ma_poly = np.insert(thetas, 0, 1.0) if self.q > 0 else np.array([1.0])
        
        sma_poly = np.zeros(self.Q * self.s + 1)
        sma_poly[0] = 1.0
        for i in range(1, self.Q + 1):
            sma_poly[i * self.s] = Thetas[i-1]
            
        full_ma = np.polymul(ma_poly, sma_poly)
        return full_ar, full_ma

    def _objective(self, params, y):
        mu = params[0]
        
        idx = 1
        phis = params[idx : idx + self.p]; idx += self.p
        thetas = params[idx : idx + self.q]; idx += self.q
        Phis = params[idx : idx + self.P]; idx += self.P
        Thetas = params[idx : idx + self.Q]
        
        full_ar, full_ma = self._build_polynomials(phis, thetas, Phis, Thetas)
        
        resid = lfilter(full_ma, full_ar, y - mu)
        sse = np.sum(resid**2)
        
        if not np.isfinite(sse) or sse > 1e10:
            return 1e10
        return sse

    def fit(self, y):
        n_params = 1 + self.p + self.q + self.P + self.Q
        initial_guess = np.zeros(n_params)
        initial_guess[0] = np.mean(y)
        
        bounds = [(None, None)] + [(-0.99, 0.99)] * (n_params - 1)
        
        res = minimize(
            self._objective, 
            initial_guess, 
            args=(y,), 
            method='L-BFGS-B',
            bounds=bounds,
            options={'maxiter': 50, 'ftol': 1e-4}
        )
        
        self.mu = res.x[0]
        idx = 1
        self.phis = res.x[idx : idx + self.p]; idx += self.p
        self.thetas = res.x[idx : idx + self.q]; idx += self.q
        self.Phis = res.x[idx : idx + self.P]; idx += self.P
        self.Thetas = res.x[idx : idx + self.Q]
        
        full_ar, full_ma = self._build_polynomials(self.phis, self.thetas, self.Phis, self.Thetas)
        y_centered = y - self.mu
        final_resids = lfilter(full_ma, full_ar, y_centered)
        
        if self.max_ar_lag > 0:
            self.last_y_centered = np.pad(y_centered, (max(0, self.max_ar_lag - len(y_centered)), 0))[-self.max_ar_lag:]
        if self.max_ma_lag > 0:
            self.last_resids = np.pad(final_resids, (max(0, self.max_ma_lag - len(final_resids)), 0))[-self.max_ma_lag:]
            
        return self

    def predict(self, steps=1):
        preds = np.zeros(steps)
        y_hist = np.concatenate([self.last_y_centered, np.zeros(steps)])
        e_hist = np.concatenate([self.last_resids, np.zeros(steps)])
        
        full_ar, full_ma = self._build_polynomials(self.phis, self.thetas, self.Phis, self.Thetas)
        
        ar_coeffs = -full_ar[1:] 
        ma_coeffs = full_ma[1:]
        
        for h in range(steps):
            ar_sum = 0
            if len(ar_coeffs) > 0:
                for i, coef in enumerate(ar_coeffs):
                    ar_sum += coef * y_hist[self.max_ar_lag + h - (i + 1)]
                    
            ma_sum = 0
            if len(ma_coeffs) > 0:
                for j, coef in enumerate(ma_coeffs):
                    ma_sum += coef * e_hist[self.max_ma_lag + h - (j + 1)]
            
            pred_centered = ar_sum + ma_sum
            y_hist[self.max_ar_lag + h] = pred_centered
            preds[h] = self.mu + pred_centered
            
        return preds


# --- 2. CLASSE SARIMA (Gestion des Différenciations) ---

class SimpleSARIMA:
    def __init__(self, p=1, d=0, q=1, P=0, D=0, Q=0, s=1):
        self.d, self.D, self.s = int(d), int(D), int(s)
        self.sarma_model = SimpleSARMA(p=p, q=q, P=P, Q=Q, s=s)
        self.history = np.array([]) 

    def fit(self, y):
        required_len = self.d + (self.D * self.s)
        if len(y) <= required_len:
            raise ValueError(f"Pas assez de données pour différencier. Requis: {required_len}, Reçu: {len(y)}")
            
        self.history = y.copy()
        y_diff = y.copy()
        
        for _ in range(self.D):
            y_diff = y_diff[self.s:] - y_diff[:-self.s]
            
        for _ in range(self.d):
            y_diff = np.diff(y_diff)
            
        self.sarma_model.fit(y_diff)
        return self

    def predict(self, steps=1):
        diff_preds = self.sarma_model.predict(steps=steps)
        
        if self.d == 0 and self.D == 0:
            return diff_preds
            
        total_len = len(self.history) + steps
        reconstructed = np.zeros(total_len)
        reconstructed[:len(self.history)] = self.history
        
        current_diff = diff_preds.copy()
        
        if self.d > 0:
            base_series_for_d = self.history.copy()
            for _ in range(self.D):
                base_series_for_d = base_series_for_d[self.s:] - base_series_for_d[:-self.s]
                
            for _ in range(self.d):
                last_val = base_series_for_d[-1]
                current_diff = last_val + np.cumsum(current_diff)
                
        for step in range(steps):
            idx = len(self.history) + step
            val = current_diff[step]
            
            for _ in range(self.D):
                val += reconstructed[idx - self.s]
                
            reconstructed[idx] = val
            
        return reconstructed[len(self.history):]


# --- 3. WFCV & EVALUATION ---

def wfcv_sarima(y, order, s_order, step_size=50, fold_size=200):
    p, d, q = order
    P, D, Q, s = s_order
    
    y_vals = y.values
    n = len(y_vals)
    preds, truths, mse_tab = [], [], []
    
    model = SimpleSARIMA(p=p, d=d, q=q, P=P, D=D, Q=Q, s=s)
    
    for start in range(0, n - fold_size - step_size + 1, step_size):
        y_train = y_vals[start : start + fold_size]
        y_test = y_vals[start + fold_size : start + fold_size + step_size]
        
        try:
            model.fit(y_train)
            fcst = model.predict(steps=len(y_test))
            mse_tab.append(mean_squared_error(y_test, fcst))
        except Exception:
            fcst = [np.nan] * len(y_test)
            mse_tab.append(np.nan)
            
        preds.extend(fcst)
        truths.extend(y_test)
        
    return np.array(preds), np.array(truths), np.array(mse_tab)

def stats_forecasting_sarima(name_ticker, df_all, order, seasonal_base, s_val, step_size=50, fold_size=200):
    P, D, Q = seasonal_base
    s_order = (P, D, Q, s_val)
    p, d, q = order
    
    df_t = df_all[['Date', name_ticker]].rename(columns={name_ticker: 'return'})
    df_t = df_t[df_t['return'] > -1].copy()
    df_t['log_return'] = np.log1p(df_t['return']).replace([np.inf, -np.inf], np.nan)
    df_t = df_t.dropna(subset=['log_return'])
    y = df_t['log_return']
    
    y_pred, y_true, mse_tab = wfcv_sarima(y, order, s_order, step_size, fold_size)

    mask = np.isfinite(y_pred) & np.isfinite(y_true)
    if mask.sum() >= 3:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            reg = sm.OLS(y_true[mask], sm.add_constant(y_pred[mask])).fit()
            ols_r2 = float(reg.rsquared)
            ols_intercept = float(reg.params[0])
            ols_slope = float(reg.params[1])
            p_int = float(reg.pvalues[0])
            p_slope = float(reg.pvalues[1])
    else:
        ols_r2 = ols_intercept = ols_slope = p_int = p_slope = float('nan')

    return {
        'SYMBOL': name_ticker,
        'Model': f'SARIMA({p},{d},{q})({P},{D},{Q})_{s_val} Custom',
        'p': p, 'd': d, 'q': q,
        'P': P, 'D': D, 'Q': Q, 's': s_val,
        'MSE': float(np.nanmean(mse_tab)) if len(mse_tab) > 0 else np.nan,
        'OLS_R2': ols_r2,
        'OLS_Intercept': ols_intercept,
        'OLS_Slope': ols_slope,
        'OLS_P_Value_Intercept': p_int,
        'P_Value_Slope': p_slope,
        'n_obs': int(len(y)),
    }

# --- 4. EXÉCUTION PRINCIPALE ---

# --- 4. EXÉCUTION PRINCIPALE ---

if __name__ == "__main__":
    STEP_SIZE = 50
    FOLD_SIZE = 200

    # 1. Chargement des données
    df_all = pd.read_csv('Datasets/returns_all.csv')
    df_all['Date'] = pd.to_datetime(df_all['Date'], errors='coerce')
    all_tickers = df_all.columns[1:].tolist()

    # 2. Configurations à tester
    sarima_configs = [
        ((1, 0, 1), (1, 0, 1)),
        ((1, 1, 1), (0, 1, 1))
    ]
    s_values = [5, 21] 
    
    # 3. Préparation du dossier de résultats
    os.makedirs('Resultats', exist_ok=True)

    # 4. Boucles Inversées : Modèles -> Tickers
    for order, seasonal_base in sarima_configs:
        for s_val in s_values:
            
            # Extraction des paramètres pour le nom du fichier
            p, d, q = order
            P, D, Q = seasonal_base
            nom_modele = f"SARIMA({p},{d},{q})({P},{D},{Q})_{s_val}"
            
            output_file = f"Resultats/resultats_{nom_modele}_all_tickers.csv"
            rows_for_config = []
            
            # La barre de progression affiche maintenant le modèle en cours d'entraînement
            for name in tqdm(all_tickers, desc=f'Entraînement {nom_modele}'):
                res = stats_forecasting_sarima(
                    name_ticker=name, 
                    df_all=df_all, 
                    order=order, 
                    seasonal_base=seasonal_base,
                    s_val=s_val,
                    step_size=STEP_SIZE, 
                    fold_size=FOLD_SIZE
                )
                rows_for_config.append(res)
                
            # Sauvegarde directe du CSV complet pour CE set de paramètres
            df_config = pd.DataFrame(rows_for_config)
            df_config.to_csv(output_file, index=False)
            
            # Petit message de confirmation avant de passer au modèle suivant
            tqdm.write(f"✅ Fichier sauvegardé : {output_file}")
            
    print("\nTous les calculs et exports individuels sont terminés avec succès !")

Entraînement SARIMA(1,0,1)(1,0,1)_5: 100%|██████████| 1066/1066 [05:41<00:00,  3.12it/s]


✅ Fichier sauvegardé : Resultats/resultats_SARIMA(1,0,1)(1,0,1)_5_all_tickers.csv


Entraînement SARIMA(1,0,1)(1,0,1)_21: 100%|██████████| 1066/1066 [04:29<00:00,  3.95it/s]


✅ Fichier sauvegardé : Resultats/resultats_SARIMA(1,0,1)(1,0,1)_21_all_tickers.csv


Entraînement SARIMA(1,1,1)(0,1,1)_5: 100%|██████████| 1066/1066 [07:28<00:00,  2.37it/s]


✅ Fichier sauvegardé : Resultats/resultats_SARIMA(1,1,1)(0,1,1)_5_all_tickers.csv


Entraînement SARIMA(1,1,1)(0,1,1)_21: 100%|██████████| 1066/1066 [09:21<00:00,  1.90it/s]

✅ Fichier sauvegardé : Resultats/resultats_SARIMA(1,1,1)(0,1,1)_21_all_tickers.csv

Tous les calculs et exports individuels sont terminés avec succès !
